In [8]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [9]:
spark.version

'3.5.0'

In [10]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [11]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [12]:
import pandas as pd

df_zones = spark.read \
    .option("header", "true") \
    .csv('zones/taxi_zone_lookup.csv')

In [29]:
df_zones.limit(1).toPandas()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR


In [35]:
df_zones.toPandas()['Zone'].nunique()

262

In [14]:
df_zones.toPandas()['service_zone'].unique()

array(['EWR', 'Boro Zone', 'Yellow Zone', 'Airports', 'N/A'], dtype=object)

In [15]:
df.registerTempTable('yellowtable')


/home/wali/data_eng_2026/6-batch/.venv/lib/python3.10/site-packages/pyspark/sql/dataframe.py:329: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [16]:
df_zones.registerTempTable('zonetable')

## Question 1

Question 1. Install Spark and PySpark  
3.5.0

In [17]:
df = df.repartition(4)

In [18]:
df.rdd.getNumPartitions()

[Stage 7:====================================>                      (5 + 3) / 8]

4

In [19]:
df.write.parquet('pq/', mode='overwrite')

## Question 2
Read the November 2025 Yellow into a Spark Dataframe.  

Repartition the Dataframe to 4 partitions and save it to parquet.  

In [20]:
ls -lh pq/

total 98M
-rw-r--r-- 1 wali wali   0 Mar  5 10:54 _SUCCESS
-rw-r--r-- 1 wali wali 25M Mar  5 10:54 part-00000-84ba0ac7-a79a-4802-b48a-f526d2b8add5-c000.snappy.parquet
-rw-r--r-- 1 wali wali 25M Mar  5 10:54 part-00001-84ba0ac7-a79a-4802-b48a-f526d2b8add5-c000.snappy.parquet
-rw-r--r-- 1 wali wali 25M Mar  5 10:54 part-00002-84ba0ac7-a79a-4802-b48a-f526d2b8add5-c000.snappy.parquet
-rw-r--r-- 1 wali wali 25M Mar  5 10:54 part-00003-84ba0ac7-a79a-4802-b48a-f526d2b8add5-c000.snappy.parquet


## What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? 
Select the answer which most closely matches.  
- 6MB  
- **25MB**   
- 75MB   
- 100MB  

## Question 3

In [21]:
df_result = spark.sql("""
SELECT 
    count(1) as total_trips
FROM
    yellowtable
where DATE(tpep_pickup_datetime) = '2025-11-15';

    
""")

In [22]:
df_result.show()

+-----------+
|total_trips|
+-----------+
|     162604|
+-----------+




How many taxi trips were there on the 15th of November?

Consider only trips that started on the 15th of November.

- 62,610  
- 102,340  
- **162,604**  
- 225,768  

## Question 4  

In [23]:
df_result = spark.sql("""
SELECT 
    (UNIX_TIMESTAMP(tpep_dropoff_datetime)- 
    UNIX_TIMESTAMP(tpep_pickup_datetime)) 
    /3600 
    as trip_duration_hours
FROM
    yellowtable

order by 
    trip_duration_hours desc

limit 1
    
""")
df_result.show()

[Stage 14:====================================>                     (5 + 3) / 8]

+-------------------+
|trip_duration_hours|
+-------------------+
|  90.64666666666666|
+-------------------+




What is the length of the longest trip in the dataset in hours?  

- 22.7  
- 58.2  
- **90.6**    
- 134.5    

## Question 5


Spark's User Interface which shows the application's dashboard runs on which local port?

- 80  
- 443  
- **4040**    
- 8080  

## Question 6: Least frequent pickup location zone

In [43]:
df_result = spark.sql("""
SELECT 
    z.zone, 
    COUNT(1) AS pickup_count
FROM 
    `zonetable` z
LEFT JOIN 
    `yellowtable` t 
    ON z.LocationID = t.PULocationID
GROUP BY 
    1
ORDER BY 
    pickup_count ASC
LIMIT 5;
    
""")
df_result.show()

[Stage 64:>                                                         (0 + 1) / 1]

+--------------------+------------+
|                zone|pickup_count|
+--------------------+------------+
|Charleston/Totten...|           1|
|     Freshkills Park|           1|
|    Great Kills Park|           1|
|Eltingville/Annad...|           1|
|       Arden Heights|           1|
+--------------------+------------+




Load the zone lookup data into a temp view in Spark:  

wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv  

Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?  

Governor's Island/Ellis Island/Liberty Island  
**Arden Heights**   
Rikers Island  
Jamaica Bay  